<a href="https://colab.research.google.com/github/PriscilaForcel/DOUTORADO/blob/main/01_BR_UNIVERSO/BR_UNIVERSO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Censo Demográfico 2022, Construção de banco nacional por setor censitário**

**Objetivo:** Desenvolver um procedimento para aquisição, padronização, integração e validação dos resultados do Universo do Censo Demográfico 2022, divulgados pelo Instituto Brasileiro de Geografia e Estatística (IBGE).

**Unidade territorial:** Setor censitário.

**Abrangência:** Brasil, 27 Unidades da Federação.

**Fonte:** IBGE - Censo Demográfico 2022, Agregados por Setores Censitários.

**Conjuntos temáticos:** Básico, Características do Domicílio (Partes 1, 2 e 3), Demografia, Parentesco, Cor ou Raça e Alfabetização.

**Produto esperado:** Banco nacional em formato Parquet, com identificadores territoriais preservados, variáveis identificadas por tema e relatório de validação.

**Aplicação:** Subsidiar pesquisas sobre necessidades habitacionais e desigualdades socioespaciais, incluindo análises voltadas à estimativa e à desagregação territorial do déficit habitacional.

*1. Configuração do ambiente*

Preparar o ambiente para o processamento dos dados do Censo Demográfico 2022. São instaladas e importadas as bibliotecas utilizadas nas etapas de aquisição, tratamento, integração e validação das bases.

Também são identificadas as versões do Python e das principais bibliotecas, além da capacidade de memória RAM e do espaço disponível em disco. Essas informações auxiliam na reprodução dos procedimentos e na definição das estratégias de processamento.

In [11]:
# 1. CONFIGURAÇÃO DO AMBIENTE

# Instalar as dependências
%pip install -q "duckdb==1.3.2" openpyxl

# Importar as bibliotecas
import os
import sys
import shutil
import zipfile
import urllib.request

import pandas as pd
import duckdb

# Identificar as versões utilizadas
print("CONFIGURAÇÃO DO AMBIENTE")
print("-" * 45)

print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")
print(f"DuckDB: {duckdb.__version__}")

# Verificar os recursos computacionais
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
disco = shutil.disk_usage("/content")

print(f"\nMemória RAM total: {ram / (1024**3):.2f} GB")
print(f"Espaço livre em disco: {disco.free / (1024**3):.2f} GB")

print("\nAmbiente configurado com sucesso!")


CONFIGURAÇÃO DO AMBIENTE
---------------------------------------------
Python: 3.13.16
Pandas: 2.2.3
DuckDB: 1.3.2

Memória RAM total: 12.67 GB
Espaço livre em disco: 87.20 GB

Ambiente configurado com sucesso!


*2. Organização do diretório de trabalho*

Criar diretório temporário no ambiente do Google Colab para armazenar os arquivos utilizados durante o processamento dos dados do Censo Demográfico 2022.

O diretório será utilizado nas etapas de aquisição, extração, padronização e integração das bases. Como o armazenamento local do Colab é temporário, os arquivos necessários à preservação dos resultados serão salvos, posteriormente, no Google Drive.

A separação entre armazenamento temporário e persistente permite organizar o processamento e reduzir operações de leitura e escrita diretamente no Google Drive.

In [12]:
# 2. ORGANIZAÇÃO DO DIRETÓRIO DE TRABALHO

# Definir o diretório temporário do projeto
pasta_local = "/content/censo2022"

# Criar o diretório, caso ainda não exista
os.makedirs(pasta_local, exist_ok=True)

# Conferir sua localização
print("DIRETÓRIO DE TRABALHO")
print("-" * 45)

print("Localização:", pasta_local)
print("Diretório disponível:", os.path.isdir(pasta_local))

# Listar arquivos existentes, sem excluir nenhum
arquivos = os.listdir(pasta_local)

print("Arquivos encontrados:", len(arquivos))

for arquivo in sorted(arquivos):
    print(" -", arquivo)


DIRETÓRIO DE TRABALHO
---------------------------------------------
Localização: /content/censo2022
Diretório disponível: True
Arquivos encontrados: 0


*3. Conexão com o Google Drive*

Conectar o Google Drive ao ambiente do Google Colab para permitir o armazenamento persistente dos arquivos originais, das bases processadas e dos resultados das validações.

Diferentemente do armazenamento temporário em /content/, os arquivos salvos no Google Drive permanecem disponíveis após o encerramento da sessão.

A conexão exige autorização do usuário e não concede acesso público aos arquivos armazenados.

In [13]:
# 3. CONEXÃO COM O GOOGLE DRIVE

from google.colab import drive

# Conectar o Google Drive ao ambiente Colab
drive.mount("/content/drive")

# Confirmar a conexão
print("Google Drive conectado com sucesso!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive conectado com sucesso!


*4. Organização dos diretórios no Google Drive*

Definir estrutura padronizada de diretórios para o armazenamento persistente dos dados do Censo Demográfico 2022 no Google Drive do usuário.

Os diretórios são organizados para separar os arquivos originais disponibilizados pelo IBGE, os dados processados e os resultados das validações.

Os arquivos originais serão obtidos diretamente das fontes oficiais do IBGE e armazenados para reutilização em execuções posteriores, evitando downloads e processamentos desnecessários.

Essa organização permite a reprodução do procedimento por outros pesquisadores, sem depender de arquivos ou diretórios pessoais.

In [15]:
# 4. ORGANIZAÇÃO DOS DIRETÓRIOS NO GOOGLE DRIVE


# Diretório principal do projeto
# Cada pesquisador pode adaptar este caminho ao seu Google Drive.

pasta_drive = (
    "/content/drive/MyDrive/DOUTORADO/IBGE_UNIVERSO"
)


# Diretórios de armazenamento
pasta_originais = os.path.join(
    pasta_drive, "DADOS_ORIGINAIS_2022"
)

pasta_tratados = os.path.join(
    pasta_drive, "DADOS_TRATADOS_2022"
)

pasta_resultados = os.path.join(
    pasta_drive, "RESULTADOS_2022"
)

# Criar os diretórios, caso não existam
diretorios = {
    "Projeto": pasta_drive,
    "Arquivos originais": pasta_originais,
    "Dados tratados": pasta_tratados,
    "Resultados": pasta_resultados
}

for nome, caminho in diretorios.items():
    os.makedirs(caminho, exist_ok=True)

# Verificar os diretórios
print("DIRETÓRIOS CONFIGURADOS")
print("-" * 50)

for nome, caminho in diretorios.items():
    status = "OK" if os.path.isdir(caminho) else "ERRO"
    print(f"{nome}: {status}")
    print(f"  {caminho}\n")


DIRETÓRIOS CONFIGURADOS
--------------------------------------------------
Projeto: OK
  /content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO

Arquivos originais: OK
  /content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/DADOS_ORIGINAIS_2022

Dados tratados: OK
  /content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/DADOS_TRATADOS_2022

Resultados: OK
  /content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/RESULTADOS_2022



*5. Aquisição dos arquivos originais do IBGE*

Realizar a aquisição dos arquivos nacionais dos Agregados por Setores Censitários do Censo Demográfico 2022, disponibilizados pelo Instituto Brasileiro de Geografia e Estatística (IBGE).

*São selecionados oito conjuntos:* Básico, Características do Domicílio (Partes 1, 2 e 3), Demografia, Parentesco, Cor ou Raça e Alfabetização. Os conjuntos específicos de populações indígenas e quilombolas e os dados de óbitos não integram esta versão da pesquisa.

O procedimento verifica a existência e a integridade dos arquivos compactados antes de realizar novos downloads. Os arquivos originais são preservados, permitindo sua reutilização e a rastreabilidade das fontes.

*Fonte:* IBGE, Censo Demográfico 2022, Agregados por Setores Censitários, arquivos nacionais em formato CSV compactado.

In [16]:
# 5. AQUISIÇÃO DOS ARQUIVOS ORIGINAIS DO IBGE

# Endereço oficial dos arquivos CSV compactados
url_base = (
    "https://ftp.ibge.gov.br/Censos/"
    "Censo_Demografico_2022/"
    "Agregados_por_Setores_Censitarios/"
    "Agregados_por_Setor_csv/"
)

# Conjuntos selecionados para a pesquisa
arquivos_ibge = {
    "BASICO": "Agregados_por_setores_basico_BR_20260520.zip",
    "DOM1": "Agregados_por_setores_caracteristicas_domicilio1_BR_20261021.zip",
    "DOM2": "Agregados_por_setores_caracteristicas_domicilio2_BR_20250417.zip",
    "DOM3": "Agregados_por_setores_caracteristicas_domicilio3_BR_20250417.zip",
    "DEM": "Agregados_por_setores_demografia_BR.zip",
    "PAR": "Agregados_por_setores_parentesco_BR.zip",
    "COR": "Agregados_por_setores_cor_ou_raca_BR.zip",
    "ALF": "Agregados_por_setores_alfabetizacao_BR.zip"
}

# Verificar a integridade de um arquivo ZIP
def zip_valido(caminho):
    if not os.path.isfile(caminho):
        return False

    try:
        with zipfile.ZipFile(caminho, "r") as arquivo:
            return arquivo.testzip() is None
    except (OSError, zipfile.BadZipFile):
        return False


print("AQUISIÇÃO DOS DADOS DO IBGE")
print("-" * 55)

for tema, nome_arquivo in arquivos_ibge.items():

    destino = os.path.join(pasta_originais, nome_arquivo)

    # Reutilizar arquivos existentes e válidos
    if zip_valido(destino):
        print(f"{tema}: arquivo existente e válido.")
        continue

    # Baixar o arquivo para o armazenamento temporário
    print(f"{tema}: iniciando download...")

    temporario = os.path.join(
        pasta_local, nome_arquivo + ".parcial"
    )

    destino_parcial = destino + ".parcial"

    try:
        urllib.request.urlretrieve(
            url_base + nome_arquivo,
            temporario
        )

        # Validar o arquivo baixado
        if not zip_valido(temporario):
            raise ValueError("O arquivo ZIP está corrompido.")

        # Salvar no Google Drive
        shutil.copy2(temporario, destino_parcial)
        os.replace(destino_parcial, destino)

        tamanho_mb = os.path.getsize(destino) / (1024**2)

        print(f"  Concluído: {tamanho_mb:.2f} MB")

    except Exception as erro:
        print(f"  ERRO ao obter {tema}: {erro}")
        raise

    finally:
        # Remover somente arquivos temporários
        for caminho in [temporario, destino_parcial]:
            if os.path.exists(caminho):
                os.remove(caminho)

print("\nAquisição concluída!")


AQUISIÇÃO DOS DADOS DO IBGE
-------------------------------------------------------
BASICO: iniciando download...
  Concluído: 14.70 MB
DOM1: iniciando download...
  Concluído: 23.08 MB
DOM2: iniciando download...
  Concluído: 80.08 MB
DOM3: iniciando download...
  Concluído: 50.27 MB
DEM: iniciando download...
  Concluído: 21.51 MB
PAR: iniciando download...
  Concluído: 60.02 MB
COR: iniciando download...
  Concluído: 41.81 MB
ALF: iniciando download...
  Concluído: 135.55 MB

Aquisição concluída!


*6. Inspeção dos arquivos compactados*

Examinar a estrutura interna dos arquivos ZIP obtidos do IBGE, identificando os arquivos CSV disponíveis, seus nomes e tamanhos descompactados.

A inspeção permite verificar a organização dos conjuntos temáticos antes da leitura e conversão para Parquet, sem extrair os arquivos ou modificar seu conteúdo.

Os resultados serão utilizados para definir um procedimento padronizado de leitura dos dados.

In [17]:
# 6. INSPEÇÃO DOS ARQUIVOS COMPACTADOS

registros_zip = []

for tema, nome_zip in arquivos_ibge.items():

    caminho_zip = os.path.join(
        pasta_originais, nome_zip
    )

    print(f"\n{tema} | {nome_zip}")
    print("-" * 65)

    with zipfile.ZipFile(caminho_zip, "r") as arquivo:

        for item in arquivo.infolist():

            if item.is_dir():
                continue

            tamanho_mb = item.file_size / (1024**2)

            registros_zip.append({
                "Tema": tema,
                "Arquivo interno": item.filename,
                "Tamanho (MB)": round(tamanho_mb, 2)
            })

            print(f"Arquivo: {item.filename}")
            print(f"Tamanho: {tamanho_mb:.2f} MB")

# Organizar o inventário
df_inventario = pd.DataFrame(registros_zip)

print("\nRESUMO DO INVENTÁRIO")
print("-" * 65)

print(f"Conjuntos temáticos: {len(arquivos_ibge)}")
print(f"Arquivos internos: {len(df_inventario)}")

display(df_inventario)



BASICO | Agregados_por_setores_basico_BR_20260520.zip
-----------------------------------------------------------------
Arquivo: Agregados_por_setores_basico_BR.csv
Tamanho: 134.44 MB

DOM1 | Agregados_por_setores_caracteristicas_domicilio1_BR_20261021.zip
-----------------------------------------------------------------
Arquivo: Agregados_por_setores_caracteristicas_domicilio1_BR_20261021.csv
Tamanho: 177.14 MB

DOM2 | Agregados_por_setores_caracteristicas_domicilio2_BR_20250417.zip
-----------------------------------------------------------------
Arquivo: Agregados_por_setores_caracteristicas_domicilio2_BR_20250417.csv
Tamanho: 747.30 MB

DOM3 | Agregados_por_setores_caracteristicas_domicilio3_BR_20250417.zip
-----------------------------------------------------------------
Arquivo: Agregados_por_setores_caracteristicas_domicilio3_BR_20250417.csv
Tamanho: 290.91 MB

DEM | Agregados_por_setores_demografia_BR.zip
-----------------------------------------------------------------
Arquiv

,Tema,Arquivo interno,Tamanho (MB)
0,BASICO,Agregados_por_setores_basico_BR.csv,134.44
1,DOM1,Agregados_por_setores_caracteristicas_domicili...,177.14
2,DOM2,Agregados_por_setores_caracteristicas_domicili...,747.30
3,DOM3,Agregados_por_setores_caracteristicas_domicili...,290.91
4,DEM,Agregados_por_setores_demografia_BR.csv,84.81
5,PAR,Agregados_por_setores_parentesco_BR.csv,345.55
6,COR,Agregados_por_setores_cor_ou_raca_BR.csv,192.11
7,ALF,Agregados_por_setores_alfabetizacao_BR.csv,701.06


*7. Identificação da estrutura dos arquivos CSV*

Examinar a estrutura dos oito arquivos CSV nacionais, considerando a codificação dos caracteres, o delimitador, os nomes das colunas e a organização dos identificadores territoriais.

A leitura é realizada diretamente dos arquivos ZIP, utilizando apenas o cabeçalho e uma pequena amostra dos registros, sem carregar integralmente as bases na memória.

Esse diagnóstico orienta a padronização dos arquivos e a posterior conversão para Parquet, preservando os identificadores territoriais e os valores originais das variáveis.

In [18]:
# 7. IDENTIFICAÇÃO DA ESTRUTURA DOS ARQUIVOS CSV

import csv
import io

diagnostico_csv = []

for tema, nome_zip in arquivos_ibge.items():

    caminho_zip = os.path.join(
        pasta_originais, nome_zip
    )

    with zipfile.ZipFile(caminho_zip, "r") as arquivo:

        # Selecionar o CSV dentro do ZIP
        arquivos_csv = [
            nome for nome in arquivo.namelist()
            if nome.lower().endswith(".csv")
        ]

        if len(arquivos_csv) != 1:
            raise ValueError(
                f"{tema}: esperado 1 CSV, encontrados "
                f"{len(arquivos_csv)}."
            )

        nome_csv = arquivos_csv[0]

        # Ler uma pequena amostra de bytes
        with arquivo.open(nome_csv) as dados:
            amostra = dados.read(65536)

        # Identificar uma codificação compatível
        codificacao = None

        for tentativa in ["utf-8-sig", "cp1252", "latin-1"]:
            try:
                texto = amostra.decode(tentativa)
                codificacao = tentativa
                break
            except UnicodeDecodeError:
                continue

        # Identificar o delimitador no cabeçalho
        primeira_linha = texto.splitlines()[0]

        delimitador = max(
            [";", ",", "\t", "|"],
            key=lambda sep: primeira_linha.count(sep)
        )

        # Ler cabeçalho e duas linhas de exemplo
        with arquivo.open(nome_csv) as dados:
            with io.TextIOWrapper(
                dados, encoding=codificacao, newline=""
            ) as texto_csv:

                leitor = csv.reader(
                    texto_csv, delimiter=delimitador
                )

                colunas = next(leitor)
                exemplos = []

                for _ in range(2):
                    linha = next(leitor, None)
                    if linha is None:
                        break
                    exemplos.append(linha[:5])

        diagnostico_csv.append({
            "Tema": tema,
            "Codificação": codificacao,
            "Delimitador": repr(delimitador),
            "Colunas": len(colunas),
            "Primeira coluna": colunas[0],
            "Exemplo": exemplos
        })

# Apresentar resultados
df_diagnostico = pd.DataFrame(diagnostico_csv)

print("DIAGNÓSTICO DOS ARQUIVOS CSV")
print("-" * 60)

display(
    df_diagnostico.drop(columns=["Exemplo"])
)

print("\nEXEMPLOS DOS IDENTIFICADORES")
print("-" * 60)

for registro in diagnostico_csv:
    print(f"\n{registro['Tema']}")
    print("Primeiras colunas e valores:", registro["Exemplo"])


DIAGNÓSTICO DOS ARQUIVOS CSV
------------------------------------------------------------


,Tema,Codificação,Delimitador,Colunas,Primeira coluna
0,BASICO,cp1252,';',38,CD_SETOR
1,DOM1,utf-8-sig,';',90,setor
2,DOM2,utf-8-sig,';',407,setor
3,DOM3,utf-8-sig,';',149,setor
4,DEM,utf-8-sig,';',37,CD_setor
5,PAR,utf-8-sig,';',183,CD_SETOR
6,COR,utf-8-sig,';',96,CD_SETOR
7,ALF,utf-8-sig,';',363,CD_setor



EXEMPLOS DOS IDENTIFICADORES
------------------------------------------------------------

BASICO
Primeiras colunas e valores: [['110001505000002', 'Urbana', '1', '0', '0,5393102'], ['110001505000003', 'Urbana', '1', '0', '0,2362175']]

DOM1
Primeiras colunas e valores: [['110001505000002', '336', '0', '0', '336'], ['110001505000003', '208', '0', '0', '208']]

DOM2
Primeiras colunas e valores: [['110001505000002', '108', '44', '0', '179'], ['110001505000003', '60', '14', '0', '132']]

DOM3
Primeiras colunas e valores: [['110001505000002', '926', 'X', '0', '136'], ['110001505000003', '556', '0', '0', '73']]

DEM
Primeiras colunas e valores: [['110001505000002', '928', '428', '500', '30'], ['110001505000003', '556', '270', '286', '15']]

PAR
Primeiras colunas e valores: [['110001505000002', '336', '200', 'X', '149'], ['110001505000003', '208', '116', 'X', '86']]

COR
Primeiras colunas e valores: [['110001505000002', '312', '87', 'X', '514'], ['110001505000003', '189', '25', '0', '336']]

*8. Padronização dos nomes das variáveis*

Estabelecer os critérios de identificação das variáveis que comporão o banco nacional integrado.

O identificador territorial será padronizado como `CD_SETOR` em todos os conjuntos de dados, preservando os códigos originais dos setores censitários como texto.

Para evitar conflitos entre variáveis de diferentes conjuntos, os nomes das colunas temáticas receberão prefixos correspondentes às suas fontes (`DOM1`, `DOM2`, `DOM3`, `DEM`, `PAR`, `COR` e `ALF`). As variáveis do arquivo Básico permanecerão com suas denominações originais.

A padronização dos nomes será realizada sem modificar os valores das variáveis. Nesta etapa, serão verificados apenas os cabeçalhos dos arquivos, sem carregar integralmente as bases na memória.

In [19]:
# 8. PADRONIZAÇÃO DOS NOMES DAS VARIÁVEIS

import csv
import io

# Configuração dos identificadores originais
chaves_originais = {
    "BASICO": "CD_SETOR",
    "DOM1": "setor",
    "DOM2": "setor",
    "DOM3": "setor",
    "DEM": "CD_setor",
    "PAR": "CD_SETOR",
    "COR": "CD_SETOR",
    "ALF": "CD_setor"
}

# Codificações identificadas na Célula 7
codificacoes = {
    "BASICO": "cp1252",
    "DOM1": "utf-8-sig",
    "DOM2": "utf-8-sig",
    "DOM3": "utf-8-sig",
    "DEM": "utf-8-sig",
    "PAR": "utf-8-sig",
    "COR": "utf-8-sig",
    "ALF": "utf-8-sig"
}

# Armazenar a correspondência dos nomes das colunas
esquemas = {}
registros_colunas = []

for tema, nome_zip in arquivos_ibge.items():

    caminho_zip = os.path.join(
        pasta_originais, nome_zip
    )

    with zipfile.ZipFile(caminho_zip, "r") as arquivo:

        nomes_csv = [
            nome for nome in arquivo.namelist()
            if nome.lower().endswith(".csv")
        ]

        if len(nomes_csv) != 1:
            raise ValueError(f"{tema}: CSV não identificado.")

        with arquivo.open(nomes_csv[0]) as dados:
            with io.TextIOWrapper(
                dados,
                encoding=codificacoes[tema],
                newline=""
            ) as texto:
                colunas = next(csv.reader(texto, delimiter=";"))

    chave = chaves_originais[tema]

    if colunas.count(chave) != 1:
        raise ValueError(
            f"{tema}: identificador {chave} ausente ou duplicado."
        )

    if len(colunas) != len(set(colunas)):
        raise ValueError(
            f"{tema}: existem nomes de colunas duplicados."
        )

    renomear = {}

    for coluna in colunas:

        if coluna == chave:
            novo_nome = "CD_SETOR"

        elif tema == "BASICO":
            novo_nome = coluna

        else:
            novo_nome = f"{tema}_{coluna}"

        renomear[coluna] = novo_nome

        registros_colunas.append({
            "Tema": tema,
            "Nome original": coluna,
            "Nome padronizado": novo_nome
        })

    esquemas[tema] = renomear

# Construir catálogo preliminar das colunas
df_colunas = pd.DataFrame(registros_colunas)

# Verificar conflitos após padronização
nomes_finais = df_colunas[
    df_colunas["Nome padronizado"] != "CD_SETOR"
]["Nome padronizado"]

if nomes_finais.duplicated().any():
    raise ValueError(
        "Existem conflitos entre os nomes padronizados."
    )

# Conferir quantidade de colunas da integração
total_final = len(nomes_finais) + 1

print("PADRONIZAÇÃO DOS NOMES DAS VARIÁVEIS")
print("-" * 55)

print("Identificador único: CD_SETOR")
print("Colunas previstas:", total_final)
print("Conflitos entre variáveis:", nomes_finais.duplicated().sum())

assert total_final == 1356, (
    f"Esperadas 1.356 colunas, encontradas {total_final}."
)

print("\nExemplos da padronização:")
display(df_colunas.groupby("Tema").head(3))

print("\nEstrutura de colunas validada!")


PADRONIZAÇÃO DOS NOMES DAS VARIÁVEIS
-------------------------------------------------------
Identificador único: CD_SETOR
Colunas previstas: 1356
Conflitos entre variáveis: 0

Exemplos da padronização:


,Tema,Nome original,Nome padronizado
0,BASICO,CD_SETOR,CD_SETOR
1,BASICO,SITUACAO,SITUACAO
2,BASICO,CD_SIT,CD_SIT
38,DOM1,setor,CD_SETOR
39,DOM1,V00001,DOM1_V00001
40,DOM1,V00002,DOM1_V00002
128,DOM2,setor,CD_SETOR
129,DOM2,V00090,DOM2_V00090
130,DOM2,V00091,DOM2_V00091
535,DOM3,setor,CD_SETOR



Estrutura de colunas validada!


*9. Validação dos identificadores territoriais*

Verificar a integridade dos códigos dos setores censitários nos oito conjuntos de dados do Censo Demográfico 2022.

São analisados o número total de registros, a quantidade de setores distintos, a presença de códigos ausentes, a existência de duplicidades e a conformidade dos identificadores com a estrutura de 15 dígitos.

O arquivo Básico constitui a referência territorial do projeto. Assim, todos os seus setores deverão ser preservados na integração final, independentemente da disponibilidade de informações nos demais conjuntos temáticos.

A leitura é realizada em blocos para reduzir o consumo de memória, mantendo os identificadores como texto e sem modificar os arquivos originais.

In [20]:
# 9. VALIDAÇÃO DOS IDENTIFICADORES TERRITORIAIS

import io

resultados_validacao = []
setores_por_tema = {}

for tema, nome_zip in arquivos_ibge.items():

    caminho_zip = os.path.join(
        pasta_originais, nome_zip
    )

    chave = chaves_originais[tema]
    codificacao = codificacoes[tema]

    total_registros = 0
    codigos_ausentes = 0
    codigos_invalidos = 0
    setores = set()

    print(f"Verificando {tema}...")

    with zipfile.ZipFile(caminho_zip, "r") as arquivo:

        nome_csv = [
            nome for nome in arquivo.namelist()
            if nome.lower().endswith(".csv")
        ][0]

        with arquivo.open(nome_csv) as dados:

            with io.TextIOWrapper(
                dados,
                encoding=codificacao,
                newline=""
            ) as texto:

                blocos = pd.read_csv(
                    texto,
                    sep=";",
                    usecols=[chave],
                    dtype=str,
                    chunksize=50000,
                    keep_default_na=False
                )

                for bloco in blocos:

                    codigos = bloco[chave]

                    total_registros += len(codigos)

                    ausentes = codigos.eq("")
                    validos = codigos.str.fullmatch(
                        r"\d{15}"
                    )

                    codigos_ausentes += int(
                        ausentes.sum()
                    )

                    codigos_invalidos += int(
                        (~validos & ~ausentes).sum()
                    )

                    setores.update(
                        codigos[validos].tolist()
                    )

    distintos = len(setores)

    duplicidades = (
        total_registros
        - codigos_ausentes
        - codigos_invalidos
        - distintos
    )

    setores_por_tema[tema] = setores

    resultados_validacao.append({
        "Tema": tema,
        "Registros": total_registros,
        "Setores distintos": distintos,
        "Códigos ausentes": codigos_ausentes,
        "Códigos inválidos": codigos_invalidos,
        "Duplicidades": duplicidades
    })

# Organizar os resultados
df_validacao_setores = pd.DataFrame(
    resultados_validacao
)

print("\nVALIDAÇÃO DOS SETORES CENSITÁRIOS")
print("-" * 65)

display(df_validacao_setores)

# Verificar a integridade das chaves
assert (
    df_validacao_setores[
        ["Códigos ausentes", "Códigos inválidos", "Duplicidades"]
    ].to_numpy() == 0
).all(), "Foram identificados problemas nos identificadores."

print("\nIdentificadores validados!")


Verificando BASICO...
Verificando DOM1...
Verificando DOM2...
Verificando DOM3...
Verificando DEM...
Verificando PAR...
Verificando COR...
Verificando ALF...

VALIDAÇÃO DOS SETORES CENSITÁRIOS
-----------------------------------------------------------------


,Tema,Registros,Setores distintos,Códigos ausentes,Códigos inválidos,Duplicidades
0,BASICO,468099,468099,0,0,0
1,DOM1,458772,458772,0,0,0
2,DOM2,458772,458772,0,0,0
3,DOM3,458772,458772,0,0,0
4,DEM,458772,458772,0,0,0
5,PAR,458772,458772,0,0,0
6,COR,458772,458772,0,0,0
7,ALF,458772,458772,0,0,0



Identificadores validados!


*10. Verificação da cobertura territorial dos conjuntos de dados*

Comparar os identificadores territoriais dos sete conjuntos temáticos com aqueles presentes no arquivo Básico do Censo Demográfico 2022.

A análise verifica se todos os setores dos conjuntos temáticos estão contemplados no Básico, identifica os setores sem correspondência e avalia a consistência da cobertura territorial entre os diferentes conjuntos.

O arquivo Básico constitui a referência territorial da integração. Assim, os setores presentes exclusivamente nesse conjunto serão preservados na base final, com valores ausentes para as variáveis temáticas não disponibilizadas.

Essa verificação é necessária para assegurar que a integração mantenha a cobertura territorial prevista, sem exclusão ou duplicação de setores censitários.

In [21]:
# 10. VERIFICAÇÃO DA COBERTURA TERRITORIAL

# Conjunto de setores do arquivo Básico
setores_basico = setores_por_tema["BASICO"]

# Conjuntos temáticos selecionados
temas = [
    "DOM1", "DOM2", "DOM3",
    "DEM", "PAR", "COR", "ALF"
]

resultados_cobertura = []

for tema in temas:

    setores_tema = setores_por_tema[tema]

    # Setores do Básico sem correspondência temática
    somente_basico = setores_basico - setores_tema

    # Setores temáticos ausentes do Básico
    somente_tema = setores_tema - setores_basico

    # Setores presentes nos dois conjuntos
    correspondentes = setores_basico & setores_tema

    resultados_cobertura.append({
        "Tema": tema,
        "Setores correspondentes": len(correspondentes),
        "Apenas no Básico": len(somente_basico),
        "Apenas no temático": len(somente_tema)
    })

# Organizar resultados
df_cobertura = pd.DataFrame(resultados_cobertura)

print("COBERTURA TERRITORIAL DOS CONJUNTOS")
print("-" * 65)

display(df_cobertura)

# Verificar se todos os temas possuem a mesma cobertura
referencia = setores_por_tema["DOM1"]

mesma_cobertura = all(
    setores_por_tema[tema] == referencia
    for tema in temas
)

print("\nVERIFICAÇÕES")
print("-" * 65)

print("Cobertura temática idêntica:", mesma_cobertura)

print(
    "Todos os setores temáticos estão no Básico:",
    (df_cobertura["Apenas no temático"] == 0).all()
)

# Interromper caso haja inconsistências
assert mesma_cobertura, (
    "Os conjuntos temáticos apresentam coberturas diferentes."
)

assert (
    df_cobertura["Apenas no temático"] == 0
).all(), (
    "Existem setores temáticos ausentes do Básico."
)

print("\nCobertura territorial validada!")


COBERTURA TERRITORIAL DOS CONJUNTOS
-----------------------------------------------------------------


,Tema,Setores correspondentes,Apenas no Básico,Apenas no temático
0,DOM1,458772,9327,0
1,DOM2,458772,9327,0
2,DOM3,458772,9327,0
3,DEM,458772,9327,0
4,PAR,458772,9327,0
5,COR,458772,9327,0
6,ALF,458772,9327,0



VERIFICAÇÕES
-----------------------------------------------------------------
Cobertura temática idêntica: True
Todos os setores temáticos estão no Básico: True

Cobertura territorial validada!


*11. Conversão do arquivo Básico para Parquet*

Converter o arquivo Básico do Censo Demográfico 2022, disponibilizado em CSV compactado, para o formato Parquet.

O arquivo é lido diretamente do ZIP, em blocos, para limitar o consumo de memória RAM. Os identificadores territoriais e os valores das variáveis são preservados como texto, sem transformações numéricas nesta etapa.

O formato Parquet será utilizado nas etapas posteriores de integração e análise, permitindo maior eficiência na leitura de colunas e no processamento dos dados.

Após a conversão, são verificados o número de registros, a quantidade de colunas e a integridade do identificador `CD_SETOR`. O arquivo original do IBGE permanece inalterado.

In [22]:
# 11. CONVERSÃO DO ARQUIVO BÁSICO PARA PARQUET

# Biblioteca para escrita de arquivos Parquet
%pip install -q pyarrow

import pyarrow as pa
import pyarrow.parquet as pq

# Definir o conjunto a processar
tema = "BASICO"

# Localizar o ZIP original
caminho_zip = os.path.join(
    pasta_originais,
    arquivos_ibge[tema]
)

# Definir os arquivos de saída
nome_saida = "BASICO_2022.parquet"

saida_local = os.path.join(
    pasta_local,
    nome_saida
)

saida_drive = os.path.join(
    pasta_tratados,
    nome_saida
)

# Preservar a ordem original das colunas
nomes_originais = list(esquemas[tema].keys())
nomes_padronizados = list(esquemas[tema].values())

# Esquema de saída: todas as colunas como texto
schema = pa.schema([
    pa.field(nome, pa.string())
    for nome in nomes_padronizados
])

total_registros = 0
escritor = None

print("CONVERSÃO DO ARQUIVO BÁSICO")
print("-" * 55)

try:

    with zipfile.ZipFile(caminho_zip, "r") as arquivo:

        nomes_csv = [
            nome for nome in arquivo.namelist()
            if nome.lower().endswith(".csv")
        ]

        if len(nomes_csv) != 1:
            raise ValueError(
                "O ZIP deve conter exatamente um CSV."
            )

        with arquivo.open(nomes_csv[0]) as dados:

            with io.TextIOWrapper(
                dados,
                encoding=codificacoes[tema],
                newline=""
            ) as texto:

                blocos = pd.read_csv(
                    texto,
                    sep=";",
                    dtype=str,
                    chunksize=10000,
                    keep_default_na=False,
                    na_filter=False
                )

                escritor = pq.ParquetWriter(
                    saida_local,
                    schema=schema,
                    compression="zstd"
                )

                for bloco in blocos:

                    # Conferir a estrutura original
                    if list(bloco.columns) != nomes_originais:
                        raise ValueError(
                            "As colunas do CSV não correspondem "
                            "ao esquema validado."
                        )

                    # Aplicar nomes padronizados
                    bloco = bloco.rename(
                        columns=esquemas[tema]
                    )

                    # Converter o bloco para Arrow
                    tabela = pa.Table.from_pandas(
                        bloco,
                        schema=schema,
                        preserve_index=False
                    )

                    # Escrever no Parquet
                    escritor.write_table(tabela)

                    total_registros += len(bloco)

finally:
    if escritor is not None:
        escritor.close()

# Conferir o arquivo gerado
metadados = pq.ParquetFile(saida_local)

linhas = metadados.metadata.num_rows
colunas = len(metadados.schema_arrow.names)

# Validar a estrutura
assert linhas == 468099, (
    f"Quantidade inesperada de registros: {linhas}"
)

assert colunas == 38, (
    f"Quantidade inesperada de colunas: {colunas}"
)

assert total_registros == linhas

# Validar a unicidade da chave
con_validacao = duckdb.connect(":memory:")

try:
    consulta = con_validacao.execute(
        """
        SELECT
            COUNT(*) AS total,
            COUNT(DISTINCT CD_SETOR) AS distintos
        FROM read_parquet(?)
        """,
        [saida_local]
    ).fetchone()
finally:
    con_validacao.close()

assert consulta == (468099, 468099), (
    "Os identificadores não são únicos."
)

# Salvar no Google Drive
saida_parcial = saida_drive + ".parcial"

shutil.copy2(saida_local, saida_parcial)

# Verificar a cópia antes de torná-la definitiva
copia = pq.ParquetFile(saida_parcial)

assert copia.metadata.num_rows == linhas
assert len(copia.schema_arrow.names) == colunas

os.replace(saida_parcial, saida_drive)

print("\nARQUIVO PARQUET VALIDADO")
print("-" * 55)
print("Registros:", linhas)
print("Colunas:", colunas)
print("Setores distintos:", consulta[1])

print(
    "Tamanho:",
    f"{os.path.getsize(saida_drive) / (1024**2):.2f} MB"
)

print("Arquivo salvo em:", saida_drive)
print("\nConversão concluída com sucesso!")


CONVERSÃO DO ARQUIVO BÁSICO
-------------------------------------------------------

ARQUIVO PARQUET VALIDADO
-------------------------------------------------------
Registros: 468099
Colunas: 38
Setores distintos: 468099
Tamanho: 10.38 MB
Arquivo salvo em: /content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/DADOS_TRATADOS_2022/BASICO_2022.parquet

Conversão concluída com sucesso!


*12. Conversão dos conjuntos temáticos para Parquet*

Converter os sete conjuntos temáticos selecionados do Censo Demográfico 2022 para o formato Parquet: Características do Domicílio (Partes 1, 2 e 3), Demografia, Parentesco, Cor ou Raça e Alfabetização.

Os arquivos CSV são lidos diretamente de seus respectivos ZIPs, em blocos de 10.000 registros, sem carregar integralmente as bases na memória.

Durante a conversão, os identificadores territoriais são padronizados como `CD_SETOR`, e as variáveis recebem prefixos correspondentes aos conjuntos de origem. Os valores originais são preservados como texto, inclusive aqueles protegidos por sigilo estatístico (`X`).

Cada arquivo convertido é validado quanto ao número de registros, à quantidade de colunas e à unicidade dos identificadores. Os resultados são armazenados separadamente para posterior integração nacional.

Os arquivos originais do IBGE permanecem inalterados.

In [23]:
# 12. CONVERSÃO DOS CONJUNTOS TEMÁTICOS PARA PARQUET

temas_tematicos = [
    "DOM1", "DOM2", "DOM3",
    "DEM", "PAR", "COR", "ALF"
]

resultados_conversao = []

for tema in temas_tematicos:

    print(f"\nPROCESSANDO {tema}")
    print("-" * 55)

    # Localizar o arquivo original e definir as saídas
    caminho_zip = os.path.join(
        pasta_originais, arquivos_ibge[tema]
    )

    nome_saida = f"{tema}_2022.parquet"

    saida_local = os.path.join(
        pasta_local, nome_saida
    )

    saida_drive = os.path.join(
        pasta_tratados, nome_saida
    )

    saida_parcial = saida_drive + ".parcial"

    # Definir a estrutura padronizada
    nomes_originais = list(esquemas[tema].keys())
    nomes_padronizados = list(esquemas[tema].values())

    schema = pa.schema([
        pa.field(nome, pa.string())
        for nome in nomes_padronizados
    ])

    # Quantidades esperadas da validação anterior
    esperado_linhas = len(setores_por_tema[tema])
    esperado_colunas = len(nomes_padronizados)

    escritor = None
    total_lido = 0

    try:
        # Ler diretamente o CSV contido no ZIP
        with zipfile.ZipFile(caminho_zip, "r") as arquivo:

            nomes_csv = [
                nome for nome in arquivo.namelist()
                if nome.lower().endswith(".csv")
            ]

            if len(nomes_csv) != 1:
                raise ValueError(
                    f"{tema}: esperado exatamente um CSV."
                )

            with arquivo.open(nomes_csv[0]) as dados:

                with io.TextIOWrapper(
                    dados,
                    encoding=codificacoes[tema],
                    newline=""
                ) as texto:

                    blocos = pd.read_csv(
                        texto,
                        sep=";",
                        dtype=str,
                        chunksize=10000,
                        keep_default_na=False,
                        na_filter=False
                    )

                    escritor = pq.ParquetWriter(
                        saida_local,
                        schema=schema,
                        compression="zstd"
                    )

                    for bloco in blocos:

                        # Validar o cabeçalho original
                        if list(bloco.columns) != nomes_originais:
                            raise ValueError(
                                f"{tema}: colunas inesperadas."
                            )

                        # Padronizar os nomes
                        bloco = bloco.rename(
                            columns=esquemas[tema]
                        )

                        # Escrever o bloco no Parquet
                        tabela = pa.Table.from_pandas(
                            bloco,
                            schema=schema,
                            preserve_index=False
                        )

                        escritor.write_table(tabela)

                        total_lido += len(bloco)

        # Fechar o arquivo antes de validá-lo
        escritor.close()
        escritor = None

        # Validar registros e colunas
        metadados = pq.ParquetFile(saida_local)

        linhas = metadados.metadata.num_rows
        colunas = len(metadados.schema_arrow.names)

        if linhas != esperado_linhas:
            raise ValueError(
                f"{tema}: esperado {esperado_linhas} "
                f"registros, encontrados {linhas}."
            )

        if colunas != esperado_colunas:
            raise ValueError(
                f"{tema}: esperado {esperado_colunas} "
                f"colunas, encontradas {colunas}."
            )

        if total_lido != linhas:
            raise ValueError(
                f"{tema}: divergência na leitura dos registros."
            )

        # Validar os identificadores
        con_validacao = duckdb.connect(":memory:")

        try:
            total, distintos, invalidos = con_validacao.execute(
                """
                SELECT
                    COUNT(*),
                    COUNT(DISTINCT CD_SETOR),
                    COUNT(*) FILTER (
                        WHERE NOT regexp_full_match(
                            CD_SETOR, '[0-9]{15}'
                        )
                        OR CD_SETOR IS NULL
                    )
                FROM read_parquet(?)
                """,
                [saida_local]
            ).fetchone()
        finally:
            con_validacao.close()

        if (
            total != esperado_linhas
            or distintos != esperado_linhas
            or invalidos != 0
        ):
            raise ValueError(
                f"{tema}: identificadores inconsistentes."
            )

        # Copiar para o Drive e conferir a cópia
        shutil.copy2(saida_local, saida_parcial)

        copia = pq.ParquetFile(saida_parcial)

        if (
            copia.metadata.num_rows != linhas
            or len(copia.schema_arrow.names) != colunas
        ):
            raise ValueError(
                f"{tema}: cópia do Parquet inconsistente."
            )

        os.replace(saida_parcial, saida_drive)

        tamanho_mb = (
            os.path.getsize(saida_drive) / (1024**2)
        )

        resultados_conversao.append({
            "Tema": tema,
            "Registros": linhas,
            "Colunas": colunas,
            "Setores distintos": distintos,
            "Tamanho (MB)": round(tamanho_mb, 2)
        })

        print(f"Registros: {linhas:,}")
        print(f"Colunas: {colunas}")
        print(f"Setores distintos: {distintos:,}")
        print(f"Tamanho: {tamanho_mb:.2f} MB")
        print("Status: VALIDADO")

    finally:
        if escritor is not None:
            escritor.close()

        # Excluir somente os arquivos temporários
        for caminho in [saida_local, saida_parcial]:
            if os.path.isfile(caminho):
                os.remove(caminho)

# Resumo dos resultados
df_conversao = pd.DataFrame(resultados_conversao)

print("\nRESUMO DA CONVERSÃO DOS CONJUNTOS TEMÁTICOS")
print("-" * 65)

display(df_conversao)

print("\nConversões concluídas e validadas!")



PROCESSANDO DOM1
-------------------------------------------------------
Registros: 458,772
Colunas: 90
Setores distintos: 458,772
Tamanho: 17.16 MB
Status: VALIDADO

PROCESSANDO DOM2
-------------------------------------------------------
Registros: 458,772
Colunas: 407
Setores distintos: 458,772
Tamanho: 59.08 MB
Status: VALIDADO

PROCESSANDO DOM3
-------------------------------------------------------
Registros: 458,772
Colunas: 149
Setores distintos: 458,772
Tamanho: 37.73 MB
Status: VALIDADO

PROCESSANDO DEM
-------------------------------------------------------
Registros: 458,772
Colunas: 37
Setores distintos: 458,772
Tamanho: 15.75 MB
Status: VALIDADO

PROCESSANDO PAR
-------------------------------------------------------
Registros: 458,772
Colunas: 183
Setores distintos: 458,772
Tamanho: 34.37 MB
Status: VALIDADO

PROCESSANDO COR
-------------------------------------------------------
Registros: 458,772
Colunas: 96
Setores distintos: 458,772
Tamanho: 25.37 MB
Status: VALIDAD

,Tema,Registros,Colunas,Setores distintos,Tamanho (MB)
0,DOM1,458772,90,458772,17.16
1,DOM2,458772,407,458772,59.08
2,DOM3,458772,149,458772,37.73
3,DEM,458772,37,458772,15.75
4,PAR,458772,183,458772,34.37
5,COR,458772,96,458772,25.37
6,ALF,458772,363,458772,93.14



Conversões concluídas e validadas!


*13. Inventário e validação dos arquivos Parquet*

Verificar a disponibilidade e a estrutura dos oito arquivos Parquet produzidos a partir dos dados originais do Censo Demográfico 2022.

São conferidos o número de registros, a quantidade de colunas, a presença do identificador territorial `CD_SETOR` e a correspondência entre os nomes das variáveis e o esquema previamente definido.

A validação utiliza os metadados dos arquivos Parquet, sem carregar integralmente os conjuntos na memória. Essa verificação constitui uma etapa preparatória para a integração nacional por setor censitário.

In [24]:
# 13. INVENTÁRIO E VALIDAÇÃO DOS ARQUIVOS PARQUET

# Conjuntos selecionados, na ordem da integração
temas_banco = [
    "BASICO", "DOM1", "DOM2", "DOM3",
    "DEM", "PAR", "COR", "ALF"
]

# Armazenar os caminhos para as próximas etapas
caminhos_parquet = {}
registros_inventario = []

for tema in temas_banco:

    caminho = os.path.join(
        pasta_tratados,
        f"{tema}_2022.parquet"
    )

    if not os.path.isfile(caminho):
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}"
        )

    # Consultar apenas os metadados
    parquet = pq.ParquetFile(caminho)

    linhas = parquet.metadata.num_rows
    colunas = parquet.schema_arrow.names

    # Estrutura definida na Célula 8
    colunas_esperadas = list(
        esquemas[tema].values()
    )

    linhas_esperadas = len(
        setores_por_tema[tema]
    )

    # Verificar a estrutura
    if linhas != linhas_esperadas:
        raise ValueError(
            f"{tema}: número de registros divergente."
        )

    if colunas != colunas_esperadas:
        raise ValueError(
            f"{tema}: esquema de colunas divergente."
        )

    if colunas.count("CD_SETOR") != 1:
        raise ValueError(
            f"{tema}: identificador territorial inválido."
        )

    caminhos_parquet[tema] = caminho

    registros_inventario.append({
        "Tema": tema,
        "Registros": linhas,
        "Colunas": len(colunas),
        "Tamanho (MB)": round(
            os.path.getsize(caminho) / (1024**2), 2
        ),
        "Status": "OK"
    })

# Organizar o inventário
df_parquets = pd.DataFrame(
    registros_inventario
)

print("INVENTÁRIO DOS ARQUIVOS PARQUET")
print("-" * 65)

display(df_parquets)

# Calcular o número de colunas do banco integrado
total_colunas = (
    df_parquets["Colunas"].sum()
    - (len(temas_banco) - 1)
)

print("\nESTRUTURA PREVISTA DO BANCO NACIONAL")
print("-" * 65)

print(
    "Setores censitários:",
    len(setores_por_tema["BASICO"])
)

print("Colunas previstas:", total_colunas)

assert total_colunas == 1356

print("\nArquivos disponíveis e estruturas validadas!")


INVENTÁRIO DOS ARQUIVOS PARQUET
-----------------------------------------------------------------


,Tema,Registros,Colunas,Tamanho (MB),Status
0,BASICO,468099,38,10.38,OK
1,DOM1,458772,90,17.16,OK
2,DOM2,458772,407,59.08,OK
3,DOM3,458772,149,37.73,OK
4,DEM,458772,37,15.75,OK
5,PAR,458772,183,34.37,OK
6,COR,458772,96,25.37,OK
7,ALF,458772,363,93.14,OK



ESTRUTURA PREVISTA DO BANCO NACIONAL
-----------------------------------------------------------------
Setores censitários: 468099
Colunas previstas: 1356

Arquivos disponíveis e estruturas validadas!


*14. Preparação da integração nacional por setor censitário*

Preparar os arquivos Parquet para a integração nacional dos dados do Censo Demográfico 2022.

Os oito conjuntos são copiados para o armazenamento temporário do Google Colab, a fim de reduzir as operações de leitura diretamente no Google Drive.

O arquivo Básico é utilizado para definir lotes de processamento, compostos inicialmente por até 10.000 setores censitários, ordenados pelo identificador `CD_SETOR`.

A divisão em lotes constitui uma estratégia computacional para limitar o uso da memória RAM. Ela não altera a unidade territorial de análise nem implica agregação ou exclusão de registros.

Cada lote será posteriormente integrado, validado e armazenado de forma independente, permitindo retomar o processamento sem repetir os lotes concluídos.

Nesta etapa, nenhuma junção entre os conjuntos de dados é realizada.

In [25]:
# 14. PREPARAÇÃO DA INTEGRAÇÃO NACIONAL

import math

# Diretórios temporários
pasta_integracao = os.path.join(
    pasta_local, "INTEGRACAO_NACIONAL"
)

pasta_entrada = os.path.join(
    pasta_integracao, "ENTRADA"
)

pasta_temporaria = os.path.join(
    pasta_integracao, "TEMP"
)

# Diretório persistente dos lotes
pasta_lotes = os.path.join(
    pasta_resultados, "LOTES_INTEGRACAO"
)

for caminho in [
    pasta_integracao,
    pasta_entrada,
    pasta_temporaria,
    pasta_lotes
]:
    os.makedirs(caminho, exist_ok=True)

# Verificar espaço disponível
espaco_livre = shutil.disk_usage(
    pasta_local
).free

tamanho_arquivos = sum(
    os.path.getsize(caminhos_parquet[tema])
    for tema in temas_banco
)

if espaco_livre < tamanho_arquivos * 4:
    raise RuntimeError(
        "Espaço em disco insuficiente para "
        "a preparação da integração."
    )

# Copiar arquivos para o disco local do Colab
caminhos_locais = {}

print("PREPARAÇÃO DOS ARQUIVOS")
print("-" * 55)

for tema in temas_banco:

    origem = caminhos_parquet[tema]

    destino = os.path.join(
        pasta_entrada,
        f"{tema}_2022.parquet"
    )

    # Reutilizar cópia local somente se o tamanho coincidir
    if (
        not os.path.isfile(destino)
        or os.path.getsize(destino) != os.path.getsize(origem)
    ):
        shutil.copy2(origem, destino)

    # Conferir estrutura da cópia
    parquet = pq.ParquetFile(destino)

    if parquet.metadata.num_rows != len(
        setores_por_tema[tema]
    ):
        raise ValueError(
            f"{tema}: cópia local inconsistente."
        )

    caminhos_locais[tema] = destino

    print(f"{tema}: arquivo local disponível.")

# Configurar DuckDB
con = duckdb.connect(":memory:")

con.execute("SET threads = 2")
con.execute("SET memory_limit = '4GB'")

con.execute(
    f"SET temp_directory = '{pasta_temporaria}'"
)

con.execute(
    "SET preserve_insertion_order = false"
)

# Obter os setores do arquivo Básico
setores_ordenados = con.execute(
    """
    SELECT CD_SETOR
    FROM read_parquet(?)
    ORDER BY CD_SETOR
    """,
    [caminhos_locais["BASICO"]]
).fetchnumpy()["CD_SETOR"]

# Definir tamanho e quantidade de lotes
tamanho_lote = 10000

total_setores = len(setores_ordenados)

quantidade_lotes = math.ceil(
    total_setores / tamanho_lote
)

# Construir o plano de processamento
plano_lotes = []

for indice in range(quantidade_lotes):

    inicio = indice * tamanho_lote

    fim = min(
        inicio + tamanho_lote,
        total_setores
    )

    plano_lotes.append({
        "Lote": indice + 1,
        "Primeiro setor": setores_ordenados[inicio],
        "Último setor": setores_ordenados[fim - 1],
        "Quantidade": fim - inicio
    })

df_plano_lotes = pd.DataFrame(plano_lotes)

# Validar o plano
assert total_setores == 468099

assert (
    df_plano_lotes["Quantidade"].sum()
    == total_setores
)

assert quantidade_lotes == 47

# Registrar o plano no Google Drive
caminho_plano = os.path.join(
    pasta_resultados,
    "PLANO_INTEGRACAO_2022.csv"
)

df_plano_lotes.to_csv(
    caminho_plano,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print("\nPLANO DE INTEGRAÇÃO NACIONAL")
print("-" * 55)

print("Setores:", total_setores)
print("Setores por lote:", tamanho_lote)
print("Quantidade de lotes:", quantidade_lotes)
print("Total planejado:",
      df_plano_lotes["Quantidade"].sum())

print("\nPrimeiros lotes:")
display(df_plano_lotes.head())

print("\nÚltimo lote:")
display(df_plano_lotes.tail(1))

print("\nPlano de integração preparado!")


PREPARAÇÃO DOS ARQUIVOS
-------------------------------------------------------
BASICO: arquivo local disponível.
DOM1: arquivo local disponível.
DOM2: arquivo local disponível.
DOM3: arquivo local disponível.
DEM: arquivo local disponível.
PAR: arquivo local disponível.
COR: arquivo local disponível.
ALF: arquivo local disponível.

PLANO DE INTEGRAÇÃO NACIONAL
-------------------------------------------------------
Setores: 468099
Setores por lote: 10000
Quantidade de lotes: 47
Total planejado: 468099

Primeiros lotes:


,Lote,Primeiro setor,Último setor,Quantidade
0,1,110001505000002,130260305000031,10000
1,2,130260305000032,150080005000753,10000
2,3,150080005000754,150550205000205,10000
3,4,150550205000206,172100005000505,10000
4,5,172100005000506,210850405000051,10000



Último lote:


,Lote,Primeiro setor,Último setor,Quantidade
46,47,521760905000042,530010805440143,8099



Plano de integração preparado!


*15. Teste de integração nacional - Primeira parte*

Realizar um teste de integração dos oito conjuntos de dados do Censo Demográfico 2022, utilizando os primeiros 10.000 setores censitários definidos no plano de processamento.

O arquivo Básico constitui a referência territorial. As variáveis dos sete conjuntos temáticos são incorporadas por meio de junções à esquerda (`LEFT JOIN`), utilizando exclusivamente o identificador padronizado `CD_SETOR`.

Para reduzir o consumo de memória, cada conjunto é filtrado pelos limites territoriais do lote antes da integração.

O resultado é armazenado em formato Parquet e submetido a verificações de quantidade de registros, estrutura das colunas, unicidade dos identificadores e correspondência dos dados.

A validação deste lote permitirá avaliar a estratégia antes da execução do processamento nacional completo.

In [26]:
# 15. TESTE DE INTEGRAÇÃO - PRIMEIRA PARTE

# Selecionar a primeira parte do plano
lote = df_plano_lotes.iloc[0]

primeiro_setor = str(lote["Primeiro setor"])
ultimo_setor = str(lote["Último setor"])
quantidade_esperada = int(lote["Quantidade"])

# Definir os arquivos de saída
saida_teste = os.path.join(
    pasta_integracao,
    "TESTE_LOTE_001.parquet"
)

saida_drive = os.path.join(
    pasta_lotes,
    "TESTE_LOTE_001.parquet"
)

# Proteger caminhos utilizados nas consultas SQL
def literal_sql(valor):
    return "'" + str(valor).replace("'", "''") + "'"

# Construir referências dos oito conjuntos
aliases = {
    "BASICO": "b",
    "DOM1": "d1",
    "DOM2": "d2",
    "DOM3": "d3",
    "DEM": "dem",
    "PAR": "par",
    "COR": "cor",
    "ALF": "alf"
}

temas_adicionais = [
    tema for tema in temas_banco
    if tema != "BASICO"
]

# Filtrar cada conjunto antes da junção
consultas_filtradas = {}

for tema, alias in aliases.items():

    caminho = literal_sql(caminhos_locais[tema])

    consultas_filtradas[tema] = (
        f"(SELECT * FROM read_parquet({caminho}) "
        f"WHERE CD_SETOR BETWEEN "
        f"{literal_sql(primeiro_setor)} AND "
        f"{literal_sql(ultimo_setor)}) {alias}"
    )

# Preservar todas as colunas do Básico
colunas_select = ["b.*"]

# Incorporar as variáveis temáticas sem repetir CD_SETOR
for tema in temas_adicionais:
    alias = aliases[tema]
    colunas_select.append(
        f"{alias}.* EXCLUDE (CD_SETOR)"
    )

# Construir a consulta de integração
sql_integracao = (
    "SELECT\n    "
    + ",\n    ".join(colunas_select)
    + "\nFROM "
    + consultas_filtradas["BASICO"]
)

for tema in temas_adicionais:
    sql_integracao += (
        "\nLEFT JOIN "
        + consultas_filtradas[tema]
        + " USING (CD_SETOR)"
    )

# Exportar o lote integrado para Parquet
sql_exportacao = (
    f"COPY ({sql_integracao}) "
    f"TO {literal_sql(saida_teste)} "
    "(FORMAT PARQUET, COMPRESSION ZSTD)"
)

print("TESTE DE INTEGRAÇÃO NACIONAL")
print("-" * 60)
print("Lote: 1")
print("Primeiro setor:", primeiro_setor)
print("Último setor:", ultimo_setor)
print("Setores esperados:", quantidade_esperada)

print("\nExecutando a integração...")

con.execute(sql_exportacao)

# Verificar a estrutura do arquivo gerado
metadados = pq.ParquetFile(saida_teste)

linhas = metadados.metadata.num_rows
nomes_colunas = metadados.schema_arrow.names
colunas = len(nomes_colunas)

print("\nVALIDAÇÃO ESTRUTURAL")
print("-" * 60)

print("Registros:", linhas)
print("Colunas:", colunas)

assert linhas == quantidade_esperada
assert colunas == 1356
assert len(nomes_colunas) == len(set(nomes_colunas))

# Validar identificadores e unicidade
total, distintos = con.execute(
    """
    SELECT
        COUNT(*),
        COUNT(DISTINCT CD_SETOR)
    FROM read_parquet(?)
    """,
    [saida_teste]
).fetchone()

assert total == quantidade_esperada
assert distintos == quantidade_esperada

# Comparar os códigos do lote com o arquivo Básico
sql_comparacao = f"""
WITH esperado AS (
    SELECT CD_SETOR
    FROM read_parquet(
        {literal_sql(caminhos_locais["BASICO"])}
    )
    WHERE CD_SETOR BETWEEN
        {literal_sql(primeiro_setor)}
        AND {literal_sql(ultimo_setor)}
),
obtido AS (
    SELECT CD_SETOR
    FROM read_parquet({literal_sql(saida_teste)})
)
SELECT
    (
        SELECT COUNT(*)
        FROM (
            SELECT CD_SETOR FROM esperado
            EXCEPT
            SELECT CD_SETOR FROM obtido
        )
    )
    +
    (
        SELECT COUNT(*)
        FROM (
            SELECT CD_SETOR FROM obtido
            EXCEPT
            SELECT CD_SETOR FROM esperado
        )
    )
"""

diferencas = con.execute(
    sql_comparacao
).fetchone()[0]

assert diferencas == 0, (
    "Existem diferenças entre os setores "
    "do lote e os setores do Básico."
)

print("Setores distintos:", distintos)
print("Diferenças territoriais:", diferencas)

# Conferir uma variável de cada conjunto temático
print("\nVALIDAÇÃO DA CORRESPONDÊNCIA TEMÁTICA")
print("-" * 60)

for tema in temas_adicionais:

    # Selecionar a primeira variável temática
    variavel = [
        nome for nome in esquemas[tema].values()
        if nome != "CD_SETOR"
    ][0]

    origem = literal_sql(caminhos_locais[tema])
    integrado = literal_sql(saida_teste)

    sql_verificacao = f"""
        SELECT COUNT(*)
        FROM read_parquet({integrado}) b
        LEFT JOIN read_parquet({origem}) t
            USING (CD_SETOR)
        WHERE b."{variavel}"
            IS DISTINCT FROM t."{variavel}"
    """

    divergencias = con.execute(
        sql_verificacao
    ).fetchone()[0]

    assert divergencias == 0, (
        f"{tema}: divergências na variável {variavel}."
    )

    print(f"{tema}: correspondência validada.")

# Salvar o lote de teste no Drive
saida_parcial = saida_drive + ".parcial"

shutil.copy2(saida_teste, saida_parcial)

copia = pq.ParquetFile(saida_parcial)

assert copia.metadata.num_rows == quantidade_esperada
assert len(copia.schema_arrow.names) == 1356

os.replace(saida_parcial, saida_drive)

tamanho_mb = os.path.getsize(
    saida_drive
) / (1024**2)

print("\nRESULTADO DO TESTE")
print("-" * 60)

print("Setores integrados:", linhas)
print("Colunas:", colunas)
print("Setores únicos:", distintos)
print("Tamanho:", f"{tamanho_mb:.2f} MB")
print("Arquivo:", saida_drive)

print("\nPRIMEIRO LOTE INTEGRADO E VALIDADO!")


TESTE DE INTEGRAÇÃO NACIONAL
------------------------------------------------------------
Lote: 1
Primeiro setor: 110001505000002
Último setor: 130260305000031
Setores esperados: 10000

Executando a integração...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


VALIDAÇÃO ESTRUTURAL
------------------------------------------------------------
Registros: 10000
Colunas: 1356
Setores distintos: 10000
Diferenças territoriais: 0

VALIDAÇÃO DA CORRESPONDÊNCIA TEMÁTICA
------------------------------------------------------------
DOM1: correspondência validada.
DOM2: correspondência validada.
DOM3: correspondência validada.
DEM: correspondência validada.
PAR: correspondência validada.
COR: correspondência validada.
ALF: correspondência validada.

RESULTADO DO TESTE
------------------------------------------------------------
Setores integrados: 10000
Colunas: 1356
Setores únicos: 10000
Tamanho: 6.39 MB
Arquivo: /content/drive/MyDrive/PRISCILA/#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/RESULTADOS_2022/LOTES_INTEGRACAO/TESTE_LOTE_001.parquet

PRIMEIRO LOTE INTEGRADO E VALIDADO!


*16. Integração automatizada dos lotes censitários*

Automatizar a integração dos conjuntos de dados do Censo Demográfico 2022 a partir dos lotes definidos pelo arquivo Básico.

Para cada lote, os oito conjuntos são filtrados pelos limites de `CD_SETOR` antes da realização das junções à esquerda (`LEFT JOIN`). O procedimento preserva os setores do Básico e incorpora as variáveis dos sete conjuntos temáticos.

Os arquivos integrados são validados quanto à quantidade de registros, ao número de colunas, à unicidade dos identificadores e à correspondência com os limites territoriais definidos.

Cada lote é salvo individualmente no Google Drive. Arquivos previamente concluídos somente são reutilizados após validação, permitindo retomar o processamento em caso de interrupção.

Inicialmente, serão processados três lotes para verificar o funcionamento da rotina automatizada antes de sua aplicação ao conjunto nacional.

In [27]:
# 16. INTEGRAÇÃO AUTOMATIZADA COM RETOMADA

import time

# Teste inicial: processar apenas os três primeiros lotes
numero_lotes_teste = 3

# Colunas esperadas no banco integrado
colunas_esperadas = list(
    esquemas["BASICO"].values()
)

for tema in temas_adicionais:
    colunas_esperadas.extend([
        nome for nome in esquemas[tema].values()
        if nome != "CD_SETOR"
    ])

assert len(colunas_esperadas) == 1356


def validar_lote(caminho, lote):
    """Verifica a estrutura e os setores de um lote integrado."""

    if not os.path.isfile(caminho):
        return False

    try:
        parquet = pq.ParquetFile(caminho)

        if parquet.schema_arrow.names != colunas_esperadas:
            return False

        esperado = int(lote["Quantidade"])

        if parquet.metadata.num_rows != esperado:
            return False

        total, distintos, primeiro, ultimo = con.execute(
            """
            SELECT
                COUNT(*),
                COUNT(DISTINCT CD_SETOR),
                MIN(CD_SETOR),
                MAX(CD_SETOR)
            FROM read_parquet(?)
            """,
            [caminho]
        ).fetchone()

        return (
            total == esperado
            and distintos == esperado
            and primeiro == str(lote["Primeiro setor"])
            and ultimo == str(lote["Último setor"])
        )

    except Exception:
        return False


def integrar_lote(lote):
    """Integra os oito conjuntos para um intervalo de setores."""

    numero = int(lote["Lote"])
    primeiro = str(lote["Primeiro setor"])
    ultimo = str(lote["Último setor"])

    nome = f"LOTE_{numero:03d}_2022.parquet"

    destino = os.path.join(pasta_lotes, nome)

    # Retomar sem repetir arquivos válidos
    if validar_lote(destino, lote):
        return "REUTILIZADO"

    # Reaproveitar o teste validado da Célula 15
    if numero == 1:

        arquivo_teste = os.path.join(
            pasta_lotes, "TESTE_LOTE_001.parquet"
        )

        if validar_lote(arquivo_teste, lote):

            parcial = destino + ".parcial"

            shutil.copy2(arquivo_teste, parcial)

            if not validar_lote(parcial, lote):
                raise ValueError(
                    "Falha na validação da cópia do lote 1."
                )

            os.replace(parcial, destino)
            return "REUTILIZADO DO TESTE"

    # Caminho temporário local
    local = os.path.join(pasta_integracao, nome)

    # Montar consultas com filtragem anterior à junção
    consultas = {}

    for tema, alias in aliases.items():

        caminho = literal_sql(caminhos_locais[tema])

        consultas[tema] = (
            f"(SELECT * FROM read_parquet({caminho}) "
            f"WHERE CD_SETOR BETWEEN "
            f"{literal_sql(primeiro)} AND "
            f"{literal_sql(ultimo)}) {alias}"
        )

    selecao = ["b.*"]

    for tema in temas_adicionais:
        selecao.append(
            f"{aliases[tema]}.* EXCLUDE (CD_SETOR)"
        )

    sql = (
        "SELECT " + ", ".join(selecao)
        + " FROM " + consultas["BASICO"]
    )

    for tema in temas_adicionais:
        sql += (
            " LEFT JOIN " + consultas[tema]
            + " USING (CD_SETOR)"
        )

    sql_copy = (
        f"COPY ({sql}) TO {literal_sql(local)} "
        "(FORMAT PARQUET, COMPRESSION ZSTD)"
    )

    parcial = destino + ".parcial"

    try:
        # Integração no disco local
        con.execute(sql_copy)

        if not validar_lote(local, lote):
            raise ValueError(
                f"Lote {numero}: resultado local inválido."
            )

        # Salvar e validar cópia no Drive
        shutil.copy2(local, parcial)

        if not validar_lote(parcial, lote):
            raise ValueError(
                f"Lote {numero}: cópia no Drive inválida."
            )

        os.replace(parcial, destino)

        return "INTEGRADO E VALIDADO"

    finally:
        # Excluir somente arquivos temporários
        for caminho in [local, parcial]:
            if os.path.isfile(caminho):
                os.remove(caminho)


# EXECUÇÃO CONTROLADA


print("INTEGRAÇÃO NACIONAL - EXECUÇÃO DE TESTE")
print("-" * 60)

resultados_lotes = []

for _, lote in df_plano_lotes.head(
    numero_lotes_teste
).iterrows():

    numero = int(lote["Lote"])

    inicio = time.perf_counter()

    status = integrar_lote(lote)

    duracao = time.perf_counter() - inicio

    resultados_lotes.append({
        "Lote": numero,
        "Setores": int(lote["Quantidade"]),
        "Status": status,
        "Tempo (s)": round(duracao, 2)
    })

    print(
        f"Lote {numero:03d}: {status} "
        f"| {duracao:.2f} segundos",
        flush=True
    )

# Resumo dos três primeiros lotes
df_resultados_lotes = pd.DataFrame(resultados_lotes)

print("\nRESUMO DO TESTE AUTOMATIZADO")
print("-" * 60)

display(df_resultados_lotes)

print(
    "Setores contemplados:",
    df_resultados_lotes["Setores"].sum()
)

print("\nTeste automatizado concluído!")


INTEGRAÇÃO NACIONAL - EXECUÇÃO DE TESTE
------------------------------------------------------------
Lote 001: REUTILIZADO DO TESTE | 0.13 segundos


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Lote 002: INTEGRADO E VALIDADO | 4.13 segundos


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Lote 003: INTEGRADO E VALIDADO | 3.31 segundos

RESUMO DO TESTE AUTOMATIZADO
------------------------------------------------------------


,Lote,Setores,Status,Tempo (s)
0,1,10000,REUTILIZADO DO TESTE,0.13
1,2,10000,INTEGRADO E VALIDADO,4.13
2,3,10000,INTEGRADO E VALIDADO,3.31


Setores contemplados: 30000

Teste automatizado concluído!


*17. Execução da integração nacional*

Executar a integração dos 47 lotes definidos pelo arquivo Básico do Censo Demográfico 2022.

O processamento utiliza a rotina validada anteriormente, preservando os 468.099 setores censitários de referência e incorporando as variáveis dos sete conjuntos temáticos.

Cada lote é validado e salvo individualmente no Google Drive. Lotes previamente concluídos e válidos são reutilizados, permitindo a retomada do processamento em caso de interrupção.

O resultado desta etapa corresponde a um conjunto de arquivos Parquet integrados, que serão posteriormente reunidos em um único banco nacional por setor censitário.

In [28]:
# 17. EXECUÇÃO DA INTEGRAÇÃO NACIONAL

import time

print("INTEGRAÇÃO NACIONAL - CENSO 2022")
print("-" * 65)

resultados_nacionais = []

for _, lote in df_plano_lotes.iterrows():

    numero = int(lote["Lote"])

    inicio = time.perf_counter()

    # Utilizar a função validada na Célula 16
    status = integrar_lote(lote)

    duracao = time.perf_counter() - inicio

    resultados_nacionais.append({
        "Lote": numero,
        "Setores": int(lote["Quantidade"]),
        "Status": status,
        "Tempo (s)": round(duracao, 2)
    })

    print(
        f"Lote {numero:03d}/047: {status} "
        f"| {duracao:.2f} s",
        flush=True
    )

# Organizar os resultados
df_resultados_nacionais = pd.DataFrame(
    resultados_nacionais
)

# Verificar a quantidade total de setores
total_integrado = (
    df_resultados_nacionais["Setores"].sum()
)

assert len(df_resultados_nacionais) == 47

assert total_integrado == 468099

print("\nRESUMO DO PROCESSAMENTO")
print("-" * 65)

print("Lotes concluídos:", len(df_resultados_nacionais))
print("Setores contemplados:", total_integrado)

print(
    "Tempo total:",
    f"{df_resultados_nacionais['Tempo (s)'].sum():.2f} s"
)

display(df_resultados_nacionais.tail(10))

print("\nINTEGRAÇÃO DOS LOTES CONCLUÍDA!")


INTEGRAÇÃO NACIONAL - CENSO 2022
-----------------------------------------------------------------
Lote 001/047: REUTILIZADO | 0.05 s
Lote 002/047: REUTILIZADO | 0.04 s
Lote 003/047: REUTILIZADO | 0.03 s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Lote 004/047: INTEGRADO E VALIDADO | 4.56 s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Lote 005/047: INTEGRADO E VALIDADO | 4.62 s


...

Lote 046/047: INTEGRADO E VALIDADO | 4.82 s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Lote 047/047: INTEGRADO E VALIDADO | 3.46 s

RESUMO DO PROCESSAMENTO
-----------------------------------------------------------------
Lotes concluídos: 47
Setores contemplados: 468099
Tempo total: 215.89 s


,Lote,Setores,Status,Tempo (s)
37,38,10000,INTEGRADO E VALIDADO,3.81
38,39,10000,INTEGRADO E VALIDADO,7.76
39,40,10000,INTEGRADO E VALIDADO,11.85
40,41,10000,INTEGRADO E VALIDADO,5.34
41,42,10000,INTEGRADO E VALIDADO,6.77
42,43,10000,INTEGRADO E VALIDADO,4.49
43,44,10000,INTEGRADO E VALIDADO,4.23
44,45,10000,INTEGRADO E VALIDADO,5.08
45,46,10000,INTEGRADO E VALIDADO,4.82
46,47,8099,INTEGRADO E VALIDADO,3.46



INTEGRAÇÃO DOS LOTES CONCLUÍDA!


*18. Consolidação do banco nacional por setor censitário*

Esta etapa reúne os 47 lotes previamente integrados em um único arquivo Parquet de abrangência nacional.

A consolidação é realizada com a biblioteca PyArrow, por meio da leitura sequencial dos arquivos em pequenos blocos, para limitar o consumo de memória RAM.

O procedimento preserva todas as variáveis e seus valores originais, sem realizar novas junções ou agregações territoriais.

Após a consolidação, são verificados o número de registros, a estrutura das colunas, a unicidade dos identificadores e a correspondência integral dos setores censitários com o arquivo Básico do IBGE.

O banco nacional é armazenado no Google Drive para posterior validação temática e utilização nas análises da pesquisa.

In [1]:
# 18A. RECUPERAÇÃO DOS LOTES APÓS REINÍCIO

import os
import pyarrow.parquet as pq

from google.colab import drive

# Reconectar o Google Drive
drive.mount("/content/drive")

# Diretório principal do projeto
# Configuração padrão para reprodução
pasta_drive = ("/content/drive/MyDrive/DOUTORADO/IBGE_UNIVERSO")


# Recuperar os diretórios
pasta_resultados = os.path.join(
    pasta_drive, "RESULTADOS_2022"
)

pasta_lotes = os.path.join(
    pasta_resultados, "LOTES_INTEGRACAO"
)

pasta_local = "/content/censo2022"

os.makedirs(pasta_local, exist_ok=True)

# Conferir os 47 arquivos de integração
lotes_encontrados = []
lotes_ausentes = []
total_setores = 0

print("RECUPERAÇÃO DOS LOTES")
print("-" * 55)

for numero in range(1, 48):

    nome = f"LOTE_{numero:03d}_2022.parquet"
    caminho = os.path.join(pasta_lotes, nome)

    if not os.path.isfile(caminho):
        lotes_ausentes.append(nome)
        continue

    parquet = pq.ParquetFile(caminho)

    if len(parquet.schema_arrow.names) != 1356:
        raise ValueError(
            f"{nome}: quantidade de colunas inesperada."
        )

    total_setores += parquet.metadata.num_rows
    lotes_encontrados.append(caminho)

print("Lotes encontrados:", len(lotes_encontrados))
print("Lotes ausentes:", len(lotes_ausentes))
print("Total de registros:", total_setores)

if lotes_ausentes:
    print("\nArquivos ausentes:")
    for nome in lotes_ausentes:
        print(nome)

assert len(lotes_encontrados) == 47
assert total_setores == 468099

print("\nArquivos recuperados com sucesso!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RECUPERAÇÃO DOS LOTES
-------------------------------------------------------
Lotes encontrados: 47
Lotes ausentes: 0
Total de registros: 468099

Arquivos recuperados com sucesso!


In [2]:
# 18B. CONSOLIDAÇÃO NACIONAL EM BLOCOS - PYARROW

import os
import shutil
import pyarrow.parquet as pq

# 1. Definir os arquivos de entrada e saída
nome_nacional = "UNIVERSO_2022_NACIONAL.parquet"

saida_local = os.path.join(
    pasta_local, nome_nacional
)

saida_drive = os.path.join(
    pasta_resultados, nome_nacional
)

saida_parcial = saida_drive + ".parcial"

# Arquivo Básico para validação territorial
caminho_basico = os.path.join(
    pasta_drive,
    "DADOS_TRATADOS_2022",
    "BASICO_2022.parquet"
)

# 2. Conferir os lotes antes de consolidar
assert len(lotes_encontrados) == 47
assert os.path.isfile(caminho_basico)

schema_referencia = pq.ParquetFile(
    lotes_encontrados[0]
).schema_arrow

assert len(schema_referencia.names) == 1356

# Conferir se todos possuem o mesmo esquema
for numero, caminho in enumerate(
    lotes_encontrados, start=1
):
    parquet = pq.ParquetFile(caminho)

    assert parquet.schema_arrow.equals(
        schema_referencia
    ), f"Esquema divergente no lote {numero}"

# 3. Consolidar sequencialmente
total_linhas = 0

print("CONSOLIDAÇÃO NACIONAL EM BLOCOS")
print("-" * 60)

with pq.ParquetWriter(
    saida_local,
    schema=schema_referencia,
    compression="zstd"
) as escritor:

    for numero, caminho in enumerate(
        lotes_encontrados, start=1
    ):

        parquet = pq.ParquetFile(caminho)
        linhas_lote = 0

        # Ler um pequeno bloco por vez
        for bloco in parquet.iter_batches(
            batch_size=1000,
            use_threads=False
        ):
            escritor.write_batch(bloco)
            linhas_lote += bloco.num_rows

        # Validar quantidade do lote
        esperado = 10000 if numero < 47 else 8099

        assert linhas_lote == esperado, (
            f"Lote {numero}: registros divergentes."
        )

        total_linhas += linhas_lote

        print(
            f"Lote {numero:03d}/047: "
            f"{linhas_lote:,} registros | "
            f"Acumulado: {total_linhas:,}",
            flush=True
        )

# 4. Validar o banco nacional
print("\nVALIDAÇÃO DO BANCO CONSOLIDADO")
print("-" * 60)

parquet_final = pq.ParquetFile(saida_local)

assert parquet_final.metadata.num_rows == 468099
assert parquet_final.schema_arrow.equals(
    schema_referencia
)

# Conferir todos os identificadores
setores_finais = set()
registros_verificados = 0

for bloco in parquet_final.iter_batches(
    batch_size=10000,
    columns=["CD_SETOR"]
):
    codigos = bloco.column(0).to_pylist()
    registros_verificados += len(codigos)

    setores_finais.update(codigos)

assert registros_verificados == 468099
assert len(setores_finais) == 468099

# Conferir os setores do arquivo Básico
setores_basico = set()

parquet_basico = pq.ParquetFile(caminho_basico)

for bloco in parquet_basico.iter_batches(
    batch_size=10000,
    columns=["CD_SETOR"]
):
    setores_basico.update(
        bloco.column(0).to_pylist()
    )

assert setores_finais == setores_basico, (
    "O banco nacional não corresponde "
    "aos setores do arquivo Básico."
)

print("Registros:", registros_verificados)
print("Setores distintos:", len(setores_finais))
print("Colunas:", len(schema_referencia.names))
print("Correspondência com o Básico: OK")

# 5. Salvar no Google Drive
print("\nSalvando o banco nacional...")

shutil.copy2(saida_local, saida_parcial)

# Conferir a estrutura da cópia
copia = pq.ParquetFile(saida_parcial)

assert copia.metadata.num_rows == 468099
assert copia.schema_arrow.equals(schema_referencia)

# Finalizar o arquivo no Drive
os.replace(saida_parcial, saida_drive)

tamanho_mb = (
    os.path.getsize(saida_drive) / (1024**2)
)

print("\nBANCO NACIONAL CONSOLIDADO")
print("-" * 60)

print("Registros:", registros_verificados)
print("Colunas:", len(schema_referencia.names))
print("Setores únicos:", len(setores_finais))
print("Tamanho:", f"{tamanho_mb:.2f} MB")
print("Arquivo:", saida_drive)

print("\nCONSOLIDAÇÃO NACIONAL CONCLUÍDA!")


CONSOLIDAÇÃO NACIONAL EM BLOCOS
------------------------------------------------------------
Lote 001/047: 10,000 registros | Acumulado: 10,000
Lote 002/047: 10,000 registros | Acumulado: 20,000
Lote 003/047: 10,000 registros | Acumulado: 30,000
Lote 004/047: 10,000 registros | Acumulado: 40,000
Lote 005/047: 10,000 registros | Acumulado: 50,000
Lote 006/047: 10,000 registros | Acumulado: 60,000
Lote 007/047: 10,000 registros | Acumulado: 70,000
Lote 008/047: 10,000 registros | Acumulado: 80,000
Lote 009/047: 10,000 registros | Acumulado: 90,000
Lote 010/047: 10,000 registros | Acumulado: 100,000
Lote 011/047: 10,000 registros | Acumulado: 110,000
Lote 012/047: 10,000 registros | Acumulado: 120,000
Lote 013/047: 10,000 registros | Acumulado: 130,000
Lote 014/047: 10,000 registros | Acumulado: 140,000
Lote 015/047: 10,000 registros | Acumulado: 150,000
Lote 016/047: 10,000 registros | Acumulado: 160,000
Lote 017/047: 10,000 registros | Acumulado: 170,000
Lote 018/047: 10,000 registros |

*19. Validação da cobertura temática do banco nacional*

Verificar a a correspondência territorial entre o banco nacional consolidado e os arquivos temáticos originais do IBGE.

São identificados os setores presentes exclusivamente no arquivo Básico e verificada a ausência de informações temáticas para esses registros. Também é conferida a presença das informações para os setores contemplados pelos conjuntos temáticos.

A validação distingue valores ausentes decorrentes da inexistência de registros temáticos dos valores protegidos por sigilo estatístico, identificados por X. Nenhum valor é modificado durante o procedimento.

In [3]:
# 19. VALIDAÇÃO DA COBERTURA TEMÁTICA

import os
import pyarrow.parquet as pq

# Arquivos para comparação
caminho_nacional = os.path.join(
    pasta_resultados,
    "UNIVERSO_2022_NACIONAL.parquet"
)

caminho_dom1 = os.path.join(
    pasta_drive,
    "DADOS_TRATADOS_2022",
    "DOM1_2022.parquet"
)

# Identificar setores presentes no conjunto temático
setores_tematicos = set()

arquivo_dom1 = pq.ParquetFile(caminho_dom1)

for bloco in arquivo_dom1.iter_batches(
    batch_size=10000,
    columns=["CD_SETOR"]
):
    setores_tematicos.update(
        bloco.column(0).to_pylist()
    )

# Verificar os registros no banco nacional
arquivo_nacional = pq.ParquetFile(caminho_nacional)

total_basico_exclusivo = 0
total_com_tema = 0
ausencias_inesperadas = 0
valores_inesperados = 0
total_x = 0

# Primeira variável do conjunto DOM1
variavel_teste = "DOM1_V00001"

for bloco in arquivo_nacional.iter_batches(
    batch_size=10000,
    columns=["CD_SETOR", variavel_teste]
):
    codigos = bloco.column(0).to_pylist()
    valores = bloco.column(1).to_pylist()

    for codigo, valor in zip(codigos, valores):

        if codigo not in setores_tematicos:
            total_basico_exclusivo += 1

            if valor is not None:
                valores_inesperados += 1

        else:
            total_com_tema += 1

            if valor is None:
                ausencias_inesperadas += 1

        if valor == "X":
            total_x += 1

# Exibir os resultados
print("VALIDAÇÃO DA COBERTURA TEMÁTICA")
print("-" * 60)

print("Setores com correspondência temática:", total_com_tema)
print("Setores exclusivos do Básico:", total_basico_exclusivo)
print("Ausências inesperadas:", ausencias_inesperadas)
print("Valores inesperados:", valores_inesperados)
print(f"Valores X em {variavel_teste}:", total_x)

# Verificações
assert total_com_tema == 458772
assert total_basico_exclusivo == 9327
assert ausencias_inesperadas == 0
assert valores_inesperados == 0

print("\nCobertura temática validada!")


VALIDAÇÃO DA COBERTURA TEMÁTICA
------------------------------------------------------------
Setores com correspondência temática: 458772
Setores exclusivos do Básico: 9327
Ausências inesperadas: 0
Valores inesperados: 0
Valores X em DOM1_V00001: 8684

Cobertura temática validada!


*20. Visualização do banco nacional integrado*

Esta etapa permite explorar o banco nacional do Censo Demográfico 2022 após sua consolidação.

O arquivo Parquet é consultado por meio do DuckDB, permitindo visualizar uma amostra dos registros e examinar a estrutura das variáveis sem carregar integralmente o banco na memória.

A visualização é utilizada para conferir a organização das informações por setor censitário, preservando os dados originais e a estrutura territorial do arquivo Básico.

In [4]:
# 20. VISUALIZAÇÃO DO BANCO NACIONAL

import duckdb
import os

# Localizar o banco nacional
caminho_nacional = os.path.join(
    pasta_resultados,
    "UNIVERSO_2022_NACIONAL.parquet"
)

# Abrir uma conexão com o DuckDB
con_visualizacao = duckdb.connect(":memory:")

# Criar uma visualização lógica, sem copiar os dados
con_visualizacao.execute(
    """
    CREATE VIEW censo2022 AS
    SELECT *
    FROM read_parquet(?)
    """.replace(
        "read_parquet(?)",
        f"read_parquet('{caminho_nacional}')"
    )
)

# Consultar os primeiros 20 setores e 15 colunas
colunas_amostra = list(
    con_visualizacao.execute(
        "DESCRIBE censo2022"
    ).df()["column_name"].head(15)
)

colunas_sql = ", ".join(
    f'"{coluna}"'
    for coluna in colunas_amostra
)

df_visualizacao = con_visualizacao.execute(
    f"""
    SELECT {colunas_sql}
    FROM censo2022
    ORDER BY CD_SETOR
    LIMIT 20
    """
).df()

print("VISUALIZAÇÃO DO BANCO NACIONAL")
print("-" * 60)

print("Exibindo 20 setores e 15 colunas.")
print("O banco completo permanece no arquivo Parquet.")

display(df_visualizacao)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

VISUALIZAÇÃO DO BANCO NACIONAL
------------------------------------------------------------
Exibindo 20 setores e 15 colunas.
O banco completo permanece no arquivo Parquet.


,CD_SETOR,SITUACAO,CD_SIT,CD_TIPO,AREA_KM2,CD_REGIAO,NM_REGIAO,CD_UF,NM_UF,CD_MUN,NM_MUN,CD_DIST,NM_DIST,CD_SUBDIST,NM_SUBDIST
0,110001505000002,Urbana,1,0,"0,5393102",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
1,110001505000003,Urbana,1,0,"0,2362175",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
2,110001505000004,Urbana,1,0,"0,2118666",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
3,110001505000006,Urbana,1,0,"0,5054477",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
4,110001505000007,Urbana,1,0,"0,2990424",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
5,110001505000011,Urbana,1,0,"0,1850833",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
6,110001505000012,Urbana,1,0,"0,2935376",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
7,110001505000014,Urbana,1,0,"0,5708013",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
8,110001505000015,Urbana,2,0,"0,3811151",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
9,110001505000016,Urbana,1,0,"0,1838927",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,


In [5]:
from google.colab import data_table

data_table.DataTable(
    df_visualizacao,
    include_index=False
)

,CD_SETOR,SITUACAO,CD_SIT,CD_TIPO,AREA_KM2,CD_REGIAO,NM_REGIAO,CD_UF,NM_UF,CD_MUN,NM_MUN,CD_DIST,NM_DIST,CD_SUBDIST,NM_SUBDIST
0,110001505000002,Urbana,1,0,"0,5393102",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
1,110001505000003,Urbana,1,0,"0,2362175",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
2,110001505000004,Urbana,1,0,"0,2118666",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
3,110001505000006,Urbana,1,0,"0,5054477",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
4,110001505000007,Urbana,1,0,"0,2990424",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
5,110001505000011,Urbana,1,0,"0,1850833",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
6,110001505000012,Urbana,1,0,"0,2935376",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
7,110001505000014,Urbana,1,0,"0,5708013",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
8,110001505000015,Urbana,2,0,"0,3811151",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,
9,110001505000016,Urbana,1,0,"0,1838927",1,Norte,11,Rondônia,1100015,Alta Floresta D'Oeste,110001505,Alta Floresta D'Oeste,11000150500,


In [6]:

# Visualizar variáveis de diferentes conjuntos temáticos

df_exemplo = con_visualizacao.execute("""
    SELECT
        CD_SETOR,
        SITUACAO,
        v0001,
        DOM1_V00001,
        DEM_V01006,
        PAR_V01042,
        COR_V01317,
        ALF_V00644
    FROM censo2022
    ORDER BY CD_SETOR
    LIMIT 20
""").df()

display(df_exemplo)


,CD_SETOR,SITUACAO,v0001,DOM1_V00001,DEM_V01006,PAR_V01042,COR_V01317,ALF_V00644
0,110001505000002,Urbana,928,336,928,336,312,88
1,110001505000003,Urbana,556,208,556,208,189,38
2,110001505000004,Urbana,222,85,222,85,71,26
3,110001505000006,Urbana,785,281,785,282,255,75
4,110001505000007,Urbana,748,291,748,291,229,52
5,110001505000011,Urbana,631,234,631,234,273,48
6,110001505000012,Urbana,884,329,884,329,317,76
7,110001505000014,Urbana,762,291,762,291,261,75
8,110001505000015,Urbana,267,100,267,100,105,20
9,110001505000016,Urbana,311,123,311,123,122,23


In [7]:
df_exemplo = con_visualizacao.execute("""
    SELECT
        CD_SETOR,
        SITUACAO,
        v0001,
        DOM1_V00001,
        DOM1_V00017,
        DOM1_V00020,
        DOM1_V00021,
        DOM1_V00022,
        DOM1_V00023,
        DOM1_V00024,
        DOM1_V00025,
        DOM1_V00026,
        DOM1_V00070,
        DOM1_V00071,
        DOM1_V00072,
        DOM1_V00073,
        DOM1_V00074,
        DOM1_V00075
    FROM censo2022
    ORDER BY CD_SETOR
    LIMIT 20
""").df()

display(df_exemplo)

,CD_SETOR,SITUACAO,v0001,DOM1_V00001,DOM1_V00017,DOM1_V00020,DOM1_V00021,DOM1_V00022,DOM1_V00023,DOM1_V00024,DOM1_V00025,DOM1_V00026,DOM1_V00070,DOM1_V00071,DOM1_V00072,DOM1_V00073,DOM1_V00074,DOM1_V00075
0,110001505000002,Urbana,928,336,52,62,21,8,X,X,0,0,928,136,428,500,52,113
1,110001505000003,Urbana,556,208,39,37,10,3,X,0,0,X,556,73,270,286,39,67
2,110001505000004,Urbana,222,85,16,11,8,X,0,0,0,0,222,24,108,114,16,31
3,110001505000006,Urbana,785,281,55,36,18,8,7,X,0,X,783,124,407,376,55,87
4,110001505000007,Urbana,748,291,58,54,10,X,X,0,0,0,748,102,373,375,58,93
5,110001505000011,Urbana,631,234,31,40,14,0,X,0,0,0,631,84,296,335,31,85
6,110001505000012,Urbana,884,329,53,66,17,X,0,X,0,0,884,121,403,481,53,105
7,110001505000014,Urbana,762,291,58,47,14,4,0,0,0,0,762,108,361,401,58,82
8,110001505000015,Urbana,267,100,17,19,7,X,0,0,0,0,267,41,141,126,17,35
9,110001505000016,Urbana,311,123,24,14,4,X,X,0,0,0,311,36,144,167,24,42
